# Belief-Agent v2: Advanced Multi-Agent

Combines beliefs, reflection, and negotiation in a single workflow.

In [ ]:
import json, sys
sys.path.insert(0, "..")
from belief_agent import BeliefAgent, BeliefState
from belief_agent.reflection import reflect
from belief_agent.negotiation import Goal, negotiate

## Create multi-agent team with diverse beliefs

In [ ]:
def make_llm(goal_idx, action, sat, tradeoffs, beliefs):
    """Factory for mock LLMs with different positions."""
    def _llm(messages):
        p = messages[-1]["content"]
        if "Rank the following goals" in p:
            goals_list = [g for g in p.split("\n") if '"' in g]
            return json.dumps([{"index": i, "rank": i+1} for i in range(len(goals_list))])
        if "Critique" in p or "Reflect" in p:
            return json.dumps({"critique": "Reasonable", "new_evidence": [], "new_contradictions": [], "updated_confidence": 0.6})
        return json.dumps({"goal_index": goal_idx, "action": action, "satisfaction": sat, "tradeoffs": tradeoffs})
    return _llm

researcher = BeliefAgent(llm_call=make_llm(0, "Use Python", 0.8, ["speed"]), name="Researcher")
engineer = BeliefAgent(llm_call=make_llm(1, "Use Rust", 0.9, ["learning curve"]), name="Engineer")
pm = BeliefAgent(llm_call=make_llm(2, "Use TypeScript", 0.7, ["ecosystem"]), name="PM")

researcher.adopt(BeliefState("Python has best libraries", confidence=0.9))
engineer.adopt(BeliefState("Rust has best performance", confidence=0.95))
pm.adopt(BeliefState("TypeScript has best team fit", confidence=0.8))

print("Team created with 3 agents.")
print(f"Researcher beliefs: {len(researcher.beliefs)}")
print(f"Engineer beliefs: {len(engineer.beliefs)}")
print(f"PM beliefs: {len(pm.beliefs)}")

## Reflect on beliefs

In [ ]:
for agent in [researcher, engineer, pm]:
    results = reflect(agent, agent.llm_call, depth=1)
    print(f"{agent.name}: reflected on {len(results)} belief(s)")
    for r in results:
        print(f"  {r.proposition}: {r.original_confidence:.2f} -> {r.updated_confidence:.2f}")

## Negotiate language choice

In [ ]:
goals = [
    Goal("Library ecosystem", priority=0.7),
    Goal("Performance", priority=0.9),
    Goal("Team productivity", priority=0.8),
]

result = negotiate([researcher, engineer, pm], "Language choice", goals, researcher.llm_call)
print(f"Consensus: {result.consensus_action}")
print(f"Satisfaction: {result.consensus_satisfaction:.2f}")
print(f"Unresolved: {result.unresolved_goals}")

## Final state

In [ ]:
for agent in [researcher, engineer, pm]:
    print(f"\n=== {agent.name} ===")
    for b in agent.list_beliefs():
        print(f"  {b}")